In [15]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai

In [16]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [17]:
# ----------------------------
# Configure Gemini API
# ----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [18]:
# ---------------------------
# LLM
# ---------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ---------------------------
# Tool
# ---------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


tools = [calculator]


# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_messages(
[
("system", "You are a helpful assistant. Use the calculator tool when needed."),
("placeholder", "{chat_history}"),
("human", "{input}")
]
)


# ---------------------------
# Tool binding
# ---------------------------

llm_with_tools = llm.bind_tools(tools)

In [19]:



# ---------------------------
# Chain
# ---------------------------

chain = prompt | llm_with_tools


# ---------------------------
# Memory
# ---------------------------

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)



In [ ]:

# ---------------------------
# Chat loop
# ---------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"input": query},
        config={"configurable": {"session_id": "demo"}}
    )

    print("\nAgent:", response.content)


You: what is your name

Agent: I do not have a name.

You: my name is rahul

Agent: It's nice to meet you, Rahul.

You: what is 34*89

Agent: 

You: what is the answer for 34 multiplied by 89

Agent: 

You: What is 25 * 4?

Agent: 

You: what is my name

Agent: [{'type': 'text', 'text': 'Your name is Rahul.', 'extras': {'signature': 'CqsBAb4+9vsS1941REtmtl7xlWgAp3nqPZlMdDbt9DMRXdeLtL57sImkXwMlCvlfrf3cXILA0J1Nn0EUxIQp+cXnfa0WEE+hwRBwf6eecKyl4Q8g0OhZNZsgUdcRQDaPXT1hvomFPvG7IH0G62Wsfa1EvI8pFuU0B8JYSz5zL+u2y7mPoYxc5X/u/ZyM+QgBvNLN5liOldAcX/tWvr5/yhh/KYC31xOWa5/c1Sls'}}]
